# Backgammon: the doubling cube, and dice odds

The doubling cube turns backgammon from a race into a betting game. It is a six-sided die marked 2, 4, 8, 16, 32, 64. At the start it sits in the middle at value 1, owned by no one. Before your roll you may **double**: offer to play on for twice the stakes. Your opponent must then **take** — accept, after which the cube (now showing 2) sits on their side and only they may double next — or **drop**, conceding the current stake and ending the game.

Owning the cube is valuable: it means only you can raise the stakes next. So handing it over is a real decision, and accepting it turns on a single number — your chance of winning from here.

## When to double, take, or drop

There are two types of games we will focus on: **money game** and **match** to 7. A money game is played until a certain time limit and the amount of points accumulated will decide the payout. A game to 7 is different in the sense that the outcome is binary and doubling decision have to be fine tuned because of the limit e.g factoring in gammon chances doesn't make sense if you are 1 away.

For simplicity let's focus on money game first, ignore gammons and assume cube is *dead*(i.e noone has doubled yet and stakes are at 1). Your opponent "liked" their position and decided to double the stakes. The "liking" they have is what we will scrutinize at this post.

- **Drop:** you lose 1 point, for certain.
- **Take:** you play on for 2 points. Winning with probability $p$, your equity is $2p - 2(1 - p) = 4p - 2$.

So take when taking beats dropping: $4p - 2 \ge -1$ → $p \ge \tfrac{1}{4}$. This is the famous **25% take point** — accept any double you win at least a quarter of the time. (That is the bare number, ignoring future doubles. Once the cube can come back your way, holding it is worth enough to pull the take point down to about **20%**.)

The doubler's mirror of the take point is the **cash point**. Once the opponent likes their side less than their take point — you are winning about 75%, 80% with a live cube — you double them out: they pass and you bank a point. Between roughly 50% and that cash point sits the **doubling window** — good enough to double, not so good they must drop.

Now the gammons. If you like your side *and* threaten a **gammon** (a double-stake win), doubling can backfire — the opponent just drops and you bank one point when playing on might have won two. That is **too good to double**. Gammons drag every threshold around; a rough rule folds them into one number, $m = W + G/2$, with $W$ your win chance and $G$ your gammon-win chance: double past $m = 0.5$, opponent drops past $m = 0.75$, too good past $m = 1$.

## Cube map for money

Each cell is a position: your win chance down the side, your gammon chance across the top — a gammon is a win too, so it never exceeds the win chance. Colour is the correct cube action in a money game, straight from the $m = W + G/2$ rule.

- **Yellow** — double, and the opponent has a **take**.
- **Blue** — double, and the opponent must **drop** — you cash a point.

Grey is the two ends: *no double* along the bottom, where you don't like it enough yet, and *too good* in the top-right, where you'd rather play on for the gammon than cash. The lines are the dead-cube rule; a live cube nudges the drop boundary out a touch.

In [1]:
import plotly.graph_objects as go

WINS = list(range(40, 100, 5))
GAMS = list(range(0, 46, 5))


def action(win, gammon):
    w, g = win / 100, gammon / 100
    if g > w:
        return None
    m = w + g / 2
    if m < 0.5:
        return 0
    if m <= 0.75:
        return 1
    if m <= 1.0:
        return 2
    return 3


LABELS = ["no double", "double / take", "double / drop", "too good"]
CELL = {0: "", 1: "take", 2: "drop", 3: "too good", None: ""}
z = [[action(w, g) for g in GAMS] for w in WINS]
text = [[CELL[action(w, g)] for g in GAMS] for w in WINS]

colors = ["#eef0f2", "#ffd94d", "#4da3ff", "#c2c2c2"]
scale = []
for i, c in enumerate(colors):
    scale += [[i / 4, c], [(i + 1) / 4, c]]

fig = go.Figure(go.Heatmap(
    z=z, x=[f"{g}%" for g in GAMS], y=[f"{w}%" for w in WINS],
    text=text, texttemplate="%{text}", textfont=dict(size=11),
    colorscale=scale, zmin=-0.5, zmax=3.5, xgap=1, ygap=1,
    colorbar=dict(tickvals=[0, 1, 2, 3], ticktext=LABELS, title="action")))
fig.update_layout(
    title="Money game: double / take / drop by win and gammon chance",
    xaxis_title="gammon-win chance", yaxis_title="game-win chance",
    margin=dict(t=60))
fig.show()

## Match play: factoring in the score

In a money game the take point was a fixed 25%, because a point was always worth a point — win or lose, its value never moved. A match breaks that. You are racing to a target, so a point is worth more or less depending on the score: the one that takes you from 6 to 7 wins the whole match, while a point banked at 6–2 barely moves anything. Once a point's worth floats with the score, so does the take point — 25% is no longer the answer everywhere.

So we need what a point is worth at a given score, and that is just your chance of winning the match from there. A **match equity table** (MET) is that chance written out for every score; the map below uses the classic Woolsey–Heinrich MET for a 7-point match.

The take-versus-drop comparison is the same one as in money — only the currency changed, from points to match-winning chances. Dropping locks in one score; taking splits you between winning and losing the doubled game, each landing at a different score. Read those three chances off the MET,

$$\text{TP} = \frac{a - b}{c - b}, \qquad \begin{aligned} a &= \text{drop (doubler scores 1)} \\ b &= \text{take and lose 2} \\ c &= \text{take and win 2} \end{aligned}$$

and the take point is the win rate where taking and dropping break even.

Set the two sliders to the giver's and taker's scores; the crosshair marks the cell, and its number is the taker's take point. The score pulls it both ways. At **5–5**, each side two points from the match, it climbs to about **30%** — the take is fine, but the cube dies after it (two points ends the match), so there is no redouble to lean on. A trailing taker takes tighter, near **34%** when behind (needing 4 while the doubler needs 2); the leader in the mirror takes loose, near **20%**.

Two edges the map leaves off. Reach 6, one point from the match, and the Crawford rule turns the cube off for a game; at 6–6, double match point, you always take and just try to win the game. And these take points are cubeless — where the cube stays live, like 2–2, a redouble you still own pulls them a little below what the map shows.

In [2]:
import plotly.graph_objects as go

MET_PCT = [
    [50, 70, 75, 83, 85, 90, 91, 94, 95, 97, 97, 98, 98, 99, 99],
    [30, 50, 60, 68, 75, 81, 85, 88, 91, 93, 94, 95, 96, 97, 98],
    [25, 40, 50, 59, 66, 71, 76, 80, 84, 87, 90, 92, 94, 95, 96],
    [17, 32, 41, 50, 58, 64, 70, 75, 79, 83, 86, 88, 90, 92, 93],
    [15, 25, 34, 42, 50, 57, 63, 68, 73, 77, 81, 84, 87, 89, 90],
    [10, 19, 29, 36, 43, 50, 56, 62, 67, 72, 76, 79, 82, 85, 87],
    [9, 15, 24, 30, 37, 44, 50, 56, 61, 66, 70, 74, 78, 81, 84],
    [6, 12, 20, 25, 32, 38, 44, 50, 55, 60, 65, 69, 73, 77, 80],
    [5, 9, 16, 21, 27, 33, 39, 45, 50, 55, 60, 64, 68, 72, 76],
    [3, 7, 13, 17, 23, 28, 34, 40, 45, 50, 55, 60, 64, 68, 71],
    [3, 6, 10, 14, 19, 24, 30, 35, 40, 45, 50, 55, 59, 63, 67],
    [2, 5, 8, 12, 16, 21, 26, 31, 36, 40, 45, 50, 54, 58, 62],
    [2, 4, 6, 10, 13, 18, 22, 27, 32, 36, 41, 46, 50, 54, 58],
    [1, 3, 5, 8, 11, 15, 19, 23, 28, 32, 37, 42, 46, 50, 54],
    [1, 2, 4, 7, 10, 13, 16, 20, 24, 29, 33, 38, 42, 46, 50],
]
MATCH = 7


def met(t, d):
    if t <= 0:
        return 100.0
    if d <= 0:
        return 0.0
    return float(MET_PCT[t - 1][d - 1])


def take_point(taker_score, giver_score):
    t, d = MATCH - taker_score, MATCH - giver_score
    a = met(t, d - 1)
    b = met(t, d - 2)
    c = met(t - 2, d)
    if c == b:
        return None
    return (a - b) / (c - b) * 100


scores = list(range(0, 6))
n = len(scores)
z = [[take_point(t, d) for t in scores] for d in scores]
text = [[f"{v:.0f}" if v is not None else "" for v in row] for row in z]

fig = go.Figure(go.Heatmap(
    z=z, x=[str(s) for s in scores], y=[str(s) for s in scores],
    text=text, texttemplate="%{text}%", textfont=dict(size=11),
    colorscale="YlGnBu", xgap=1, ygap=1, colorbar=dict(title="take point")))

lo, hi = -0.5, n - 0.5
start = n - 1
fig.update_layout(shapes=[
    dict(type="rect", xref="x", yref="y", x0=lo, x1=hi, y0=start - 0.5, y1=start + 0.5,
         fillcolor="rgba(0,0,0,0.20)", line=dict(width=0)),
    dict(type="rect", xref="x", yref="y", x0=start - 0.5, x1=start + 0.5, y0=lo, y1=hi,
         fillcolor="rgba(0,0,0,0.20)", line=dict(width=0)),
])

giver_steps = [dict(method="relayout", label=str(s),
                    args=[{"shapes[0].y0": i - 0.5, "shapes[0].y1": i + 0.5}])
               for i, s in enumerate(scores)]
taker_steps = [dict(method="relayout", label=str(s),
                    args=[{"shapes[1].x0": j - 0.5, "shapes[1].x1": j + 0.5}])
               for j, s in enumerate(scores)]

fig.update_layout(
    sliders=[
        dict(active=start, y=-0.02, len=0.6, currentvalue={"prefix": "cube giver score = "},
             steps=giver_steps),
        dict(active=start, y=-0.32, len=0.6, currentvalue={"prefix": "cube taker score = "},
             steps=taker_steps),
    ],
    title="Match-play take point by score (7-point match, Woolsey MET)",
    xaxis_title="cube taker — score",
    yaxis_title="cube giver — score",
    margin=dict(t=60, b=140))
fig.show()

## Dice odds, at the board

Sometimes the next roll swings your chances hard — you left a blot, and whether you get hit rides on a single number. If the cube comes at you right then, you need those odds fast. Every turn is one of 36 equally likely rolls, so the counts are quick to keep:

- **A direct shot** — a number 1 to 6 pips away — sits in **11 of 36** rolls, just over 30%.
- **Combined numbers** thin out fast: 7 has 6 shots, 8 has 6, 9 has 5, 10 has 3, 11 has 2, 12 has 3. Past 12 only doubles reach — 15, 16, 18, 20, 24 are one shot each.
- **Entering from the bar** against a board with $n$ points made: you miss only if both dice land on closed points, so $P(\text{enter}) = 1 - (n/6)^2$.
- **Doubles** are **1 in 6**; **missing** a given number is $(5/6)^2 = 25/36$, about 69%; the average roll moves **8.17 pips**.

Run them against the 25% take point:

- On the bar, entering saves the game — against a 3-point board you come in 75%, an easy take; against 5 points only 31%, still a bare take; closed out, you drop.
- A hit wins outright — a shot 6 away lands 47%, a huge swing but under 50%, so not yet a double.
- Any single number wins on the last roll — 11/36, about 31%, over the take point, so if the cube comes, take.

The helpers below build every number from the 36 rolls, so you can check any distance or board yourself.

In [3]:
DICE = [(a, b) for a in range(1, 7) for b in range(1, 7)]


def reachable(a, b):
    return {a, 2 * a, 3 * a, 4 * a} if a == b else {a, b, a + b}


def shots(distance):
    return sum(1 for a, b in DICE if distance in reachable(a, b))


def hit_prob(distance):
    return shots(distance) / 36


def enter_prob(points_closed):
    return 1 - (points_closed / 6) ** 2


for k in list(range(1, 13)) + [15, 16, 18, 20, 24]:
    print(f"hit {k:>2} pips away: {shots(k):>2}/36 = {hit_prob(k) * 100:4.1f}%")
print()
for closed in range(1, 7):
    print(f"enter vs {closed} closed points: {enter_prob(closed) * 100:5.1f}%")
print()
doubles = sum(1 for a, b in DICE if a == b)
miss = sum(1 for a, b in DICE if a != 3 and b != 3)
pips = sum(4 * a if a == b else a + b for a, b in DICE)
print(f"any double: {doubles}/36 = {doubles / 36 * 100:.1f}%")
print(f"miss a given number: {miss}/36 = {miss / 36 * 100:.1f}%")
print(f"average pips per roll: {pips / 36:.2f}")

hit  1 pips away: 11/36 = 30.6%
hit  2 pips away: 12/36 = 33.3%
hit  3 pips away: 14/36 = 38.9%
hit  4 pips away: 15/36 = 41.7%
hit  5 pips away: 15/36 = 41.7%
hit  6 pips away: 17/36 = 47.2%
hit  7 pips away:  6/36 = 16.7%
hit  8 pips away:  6/36 = 16.7%
hit  9 pips away:  5/36 = 13.9%
hit 10 pips away:  3/36 =  8.3%
hit 11 pips away:  2/36 =  5.6%
hit 12 pips away:  3/36 =  8.3%
hit 15 pips away:  1/36 =  2.8%
hit 16 pips away:  1/36 =  2.8%
hit 18 pips away:  1/36 =  2.8%
hit 20 pips away:  1/36 =  2.8%
hit 24 pips away:  1/36 =  2.8%

enter vs 1 closed points:  97.2%
enter vs 2 closed points:  88.9%
enter vs 3 closed points:  75.0%
enter vs 4 closed points:  55.6%
enter vs 5 closed points:  30.6%
enter vs 6 closed points:   0.0%

any double: 6/36 = 16.7%
miss a given number: 25/36 = 69.4%
average pips per roll: 8.17


## The one number

A double says you like your side; a take says you like it enough. The rest is just where the line sits: 25% for money, wherever the score puts it in a match, and whatever the dice say when a single roll decides it.

**[In-browser notebook →](/lite/notebooks/index.html?path=2026-09-24-backgammon-cube-and-dice.ipynb)**